# IFS ensemble: fill the NaN fields from the 2018-2022 store (v2 -> v3)

The v2 files have about 20% of their (variable, start, lead) fields completely empty (5,151 in pilot, the same fields in Denver). Re-reading the 2016-2024 store does not help: those chunks do not exist there. The separate `2018-2022-1440x721` store has them (checked: 400 of 400 sampled holes present, values identical to the main store where both have data). This notebook fills the holes that fall in 2018-2022 (2,873 of 5,151) from it and writes repaired files to `tempest_wb2_ifs_ens_v3`. The v2 folder is only read. The 2016-2017 and 2023-2024 holes (about 2,280) stay NaN: no second store covers those years.

**Order:** run cells 1-2, then cell 3 (a 12-field test, about 5 min). Only if cell 3 says `RECOVERABLE`, set `TEST_ONLY = False`, rerun cell 2, run cell 4, then cell 5.

**Time (estimate):** about 1.3 TB read inside Google. About 1 h in one notebook, about 20-25 min with 3 parts (set `PARTS = 3` and `PART = 0, 1, 2` in three copies). Retries on stubborn fields add to this.

**Several accounts:** the v2 files were split across accounts. Run this on each account against the v2 files on that account's Drive; it repairs whatever files are in its `IN` folder.

In [ ]:
!pip -q install -U gcsfs "zarr>=3" xarray netCDF4 dask
import zarr, xarray   # if this fails, or cell 2 says "unrecognized engine 'zarr'": Runtime -> Restart session, rerun cells 1-2
print("zarr", zarr.__version__, "xarray", xarray.__version__)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, glob, gc, ctypes, shutil
import numpy as np, xarray as xr, fsspec, psutil
from concurrent.futures import ThreadPoolExecutor

TEST_ONLY = True               # True: cell 3 only tests 12 fields. Set False for the full repair.
PART, PARTS = 0, 1             # parallel notebooks: PARTS = 3, and PART = 0 / 1 / 2 in the three copies
WORKERS = 2                    # reads at once (~1.5 GB peak each). Colab has 12.7 GB
TRIES = 3                      # attempts per field, with a growing wait between them
IN  = "/content/drive/MyDrive/tempest_wb2_ifs_ens_v2"   # read only
OUT = "/content/drive/MyDrive/tempest_wb2_ifs_ens_v3"   # repaired files

URL = "gs://weatherbench2/datasets/ifs_ens/2018-2022-1440x721.zarr"   # second copy of the IFS ensemble: has the chunks the 2016-2024 store lacks (identical values where both have data)
UNION = ((31.0, 40.0), (252.0, 259.0))
BOXES = {"pilot": ((31.0, 34.0), (252.0, 259.0)), "denver": ((38.0, 40.0), (254.0, 256.0))}
UPPER = {"u_component_of_wind": "u", "v_component_of_wind": "v", "temperature": "t",
         "specific_humidity": "q", "geopotential": "z"}
SURF = {"10m_u_component_of_wind": "u10", "10m_v_component_of_wind": "v10",
        "2m_temperature": "t2m", "mean_sea_level_pressure": "msl"}
LONG = {v: k for k, v in {**UPPER, **SURF}.items()}
LEVELS = [500, 700, 850]

def open_ds(fresh=False):
    return xr.open_zarr(fsspec.get_mapper(URL, token="anon", skip_instance_cache=fresh), consolidated=True, chunks=None)

ds = open_ds()
HAVE = set(ds.time.values.astype("datetime64[h]"))      # start dates in this store (2018-2022 only)
lat_up = bool(ds.latitude[1] > ds.latitude[0])
td = ds.prediction_timedelta.dtype.kind == "m"
def lat_slice(la): return slice(*la) if lat_up else slice(la[1], la[0])
def lead_value(h): return np.timedelta64(int(h), "h") if td else int(h)

files = sorted(os.path.basename(f) for f in glob.glob(f"{IN}/pilot/*_00z.nc"))
MINE = files[PART::PARTS]
os.makedirs(f"{OUT}/pilot", exist_ok=True); os.makedirs(f"{OUT}/denver", exist_ok=True)
print(len(files), "month files in IN;", len(MINE), "in this part")

def read_var(short, t, L):
    # one variable, one start, one lead = one whole-globe chunk. Returns (array or None, last error text)
    err = "read came back NaN"
    for k in range(TRIES):
        try:
            d = ds if k == 0 else open_ds(True)         # fresh connection on retries
            a = d[LONG[short]].sel(time=t, prediction_timedelta=L, latitude=lat_slice(UNION[0]), longitude=slice(*UNION[1]))
            a = (a.sel(level=LEVELS) if "level" in a.dims else a).load()
            if np.isfinite(a.values).all():
                return a.astype("float32"), ""
            err = "read came back NaN"
        except Exception as e:
            err = repr(e)[:200]
        time.sleep(5 * 2 ** k)
    return None, err

def find_holes(old):
    # (variable, start index, lead index) whose whole field is NaN in either box, and whose start date is in this store
    holes = set()
    for x in old.values():
        for v in x.data_vars:
            e = [d for d in x[v].dims if d not in ("time", "prediction_timedelta")]
            bad = x[v].isnull().any(e).transpose("time", "prediction_timedelta").values
            holes |= {(v, int(i), int(j)) for i, j in zip(*np.nonzero(bad))}
    T = old["pilot"].time.values.astype("datetime64[h]")
    return sorted(h for h in holes if T[h[1]] in HAVE)

def lead_axis(x):
    L = x.prediction_timedelta.values
    return [lead_value(h) for h in (L / np.timedelta64(1, "h")).astype(int)] if L.dtype.kind == "m" else [lead_value(h) for h in L]

### Cell 3: test 12 fields first

In [ ]:
for f0 in MINE:                    # first month in this part that has holes the 2018-2022 store can fill
    old = {b: xr.load_dataset(f"{IN}/{b}/{f0}") for b in BOXES}
    holes = find_holes(old)
    if holes:
        break
assert holes, "no fillable holes in this part's files (they may all be 2016-17 / 2023-24)"
T_, L_ = old["pilot"].time.values, lead_axis(old["pilot"])
print(f0, ":", len(holes), "holes in this file")
pick = holes[::max(1, len(holes) // 12)][:12]
t0, ok = time.time(), 0
with ThreadPoolExecutor(WORKERS) as pool:
    res = list(pool.map(lambda h: read_var(h[0], T_[h[1]], L_[h[2]]), pick))
for (v, i, j), (a, err) in zip(pick, res):
    ok += a is not None
    print("ok       " if a is not None else "STILL NaN", v, str(T_[i])[:10], "lead_idx", j, err)
print(f"{ok} of {len(pick)} recovered in {time.time() - t0:.0f} s")
print("RECOVERABLE: set TEST_ONLY = False, rerun cell 2, run cell 4" if pick and ok >= len(pick) * 0.75 else
      "STOP: most fields stay NaN, so they look like real store gaps. Send this output to Claude, do not run cell 4")
del old, res; gc.collect()

### Cell 4: repair (rerun cells 1, 2 and 4 after a disconnect; finished months are skipped)

In [ ]:
assert not TEST_ONLY, "set TEST_ONLY = False in cell 2, rerun cell 2, then run this"
todo = [f for f in MINE if not all(os.path.exists(f"{OUT}/{b}/{f}") for b in BOXES)]
print(len(todo), "months to do in this part")
t_start, n, total_fixed, total_left = time.time(), 0, 0, 0
for f in todo:
    t0 = time.time()
    old = {b: xr.load_dataset(f"{IN}/{b}/{f}") for b in BOXES}
    holes = find_holes(old)
    T_, L_ = old["pilot"].time.values, lead_axis(old["pilot"])
    with ThreadPoolExecutor(WORKERS) as pool:
        got = list(pool.map(lambda h: read_var(h[0], T_[h[1]], L_[h[2]]), holes))
    left = [(h, a_e[1]) for h, a_e in zip(holes, got) if a_e[0] is None]
    if left:
        with open(f"{OUT}/store_gaps.txt", "a") as g:
            for (v, i, j), e in left:
                g.write(f"{f} {v} {str(T_[i])[:13]} lead_idx={j} {e}" + chr(10))
    for (v, i, j), (a, e) in zip(holes, got):
        if a is None:
            continue
        for b, (la, lo) in BOXES.items():
            x = old[b]
            piece = a.sel(latitude=lat_slice(la), longitude=slice(*lo))
            x[v][dict(time=i, prediction_timedelta=j)] = piece.transpose(*[d for d in x[v].dims if d in piece.dims]).values
    for b, x in old.items():
        p = f"{OUT}/{b}/{f}"
        x.to_netcdf(p + ".part")
        os.replace(p + ".part", p)
    total_fixed += len(holes) - len(left); total_left += len(left)
    del old, got; gc.collect(); ctypes.CDLL("libc.so.6").malloc_trim(0)
    n += 1
    per = (time.time() - t_start) / n
    print(f"{f}: {len(holes) - len(left)} of {len(holes)} fields filled in {time.time() - t0:.0f} s; about {per * (len(todo) - n) / 3600:.1f} h left; RAM {psutil.virtual_memory().percent:.0f}%", flush=True)
print("ALL DONE (this part): filled", total_fixed, "still empty", total_left, "(listed in store_gaps.txt)")

### Cell 5: check

In [ ]:
for b, (la, lo) in BOXES.items():
    fs = sorted(glob.glob(f"{OUT}/{b}/*_00z.nc"))
    empty = partial = bad = 0
    for f in fs:
        x = xr.open_dataset(f)
        bad += not (set(x.data_vars) == set(UPPER.values()) | set(SURF.values()) and x.sizes["number"] == 50
                    and x.sizes["prediction_timedelta"] == 7 and list(x.level.values) == LEVELS
                    and x.sizes["latitude"] == round((la[1] - la[0]) / 0.25) + 1
                    and x.sizes["longitude"] == round((lo[1] - lo[0]) / 0.25) + 1)
        for v in x.data_vars:
            e = [d for d in x[v].dims if d not in ("time", "prediction_timedelta")]
            a, c = x[v].isnull().all(e), x[v].notnull().all(e)
            empty += int(a.sum()); partial += int((~(a | c)).sum())
        x.close()
    print(b, len(fs), "files | bad shape:", bad, "| fully empty fields left (expected about 2,280 per box, all in 2016-17 and 2023-24):", empty, "| partly empty (must be 0):", partial)

## Copy step

1. In Google Drive (web), right-click `tempest_wb2_ifs_ens_v3` -> **Download** (from each account used).
2. Unzip into **new** folders (not over the old data):
   - `tempest_wb2_ifs_ens_v3/pilot/*` -> `C:\Users\Senai\Tempest-ARKNET\data\raw\wb2_ifs_ens_v3\`
   - `tempest_wb2_ifs_ens_v3/denver/*` -> `C:\Users\Senai\Tempest-ARKNET\data\raw\wb2_ifs_ens_denver_v3\`
3. Tell Claude it is there; it will check the files and swap them in.